# RQ7 — sensitive fields and privacy-policy disclosure

A Seaborn heatmap inspired by the paper's category-by-disclosure layout. This project uses only the comparable policy-audit records that can be mapped unambiguously to its selected sensitive categories. Cell values are **parameter counts**, not percentages; the small category sizes make percentages easy to overinterpret.

In [ ]:
from collections import Counter
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
audit_dir = project_root / 'backend' / 'final_results'
assert audit_dir.is_dir(), f'Policy-audit folder not found: {audit_dir}'

# Reuse the conservative, unambiguous mapping used in bolum3_uygulama.ipynb.
sensitive_type_map = {
    'Passwords': 'Security credentials',
    'Email address': 'Personal information',
    'Name': 'Personal information',
    'Phone number': 'Personal information',
    'Other financial info': 'Finance information',
    'Purchase history': 'Finance information',
}
status_priority = ('CLEAR', 'VAGUE', 'AMBIGUOUS', 'INCORRECT')
status_names = {'CLEAR': 'Clear', 'VAGUE': 'Vague',
                'INCORRECT': 'Incorrect', 'AMBIGUOUS': 'Ambiguous'}
rows = []
for path in sorted(audit_dir.glob('*.json')):
    try:
        items = json.loads(path.read_text(encoding='utf-8'))
    except (json.JSONDecodeError, UnicodeDecodeError):
        continue
    for item in items:
        category = sensitive_type_map.get(item.get('data_type'))
        collection = item.get('collection') or []
        if category is None or not collection:
            continue  # not sensitive by our mapping, or no comparable policy text
        labels = {sentence.get('label') for sentence in collection}
        first_match = next((label for label in status_priority if label in labels), None)
        status = status_names[first_match] if first_match else 'Omitted'
        rows.append({'category': category, 'status': status})

audit = pd.DataFrame(rows)
category_order = ['Personal information', 'Finance information',
                  'Security credentials', 'Health information']
status_order = ['Clear', 'Vague', 'Incorrect', 'Ambiguous', 'Omitted']
counts = pd.crosstab(audit['category'], audit['status']).reindex(
    index=category_order, columns=status_order, fill_value=0
).fillna(0).astype(int)
print(counts)
print(f'\nMapped sensitive: {len(audit)}; omitted: {(audit.status == "Omitted").sum()}')


In [ ]:
sns.set_theme(style='white', font_scale=1.0)
row_sizes = counts.sum(axis=1)
plot_counts = counts.astype(float)
plot_counts.loc[row_sizes == 0, :] = np.nan  # no data is different from zero
plot_counts.index = [f'{category} (n={row_sizes[category]})'
                     for category in category_order]
annotations = plot_counts.apply(
    lambda column: column.map(lambda value: '' if pd.isna(value) else str(int(value)))
)

fig, ax = plt.subplots(figsize=(9.5, 4.8))
sns.heatmap(
    plot_counts, annot=annotations, fmt='', cmap='YlOrRd',
    vmin=0, vmax=counts.to_numpy().max(), mask=plot_counts.isna(),
    linewidths=0.6, linecolor='white',
    cbar_kws={'label': 'Number of audited parameters'}, ax=ax,
)
ax.text(2.5, 3.5, 'No comparable records', ha='center', va='center',
        color='dimgray', fontsize=10)
ax.set_title('Sensitive fields and privacy-policy disclosure', loc='left', pad=16)
ax.set_xlabel('Disclosure status')
ax.set_ylabel('Selected sensitive category')
ax.tick_params(axis='x', rotation=0)
ax.tick_params(axis='y', rotation=0)
fig.text(0.12, 0.01, '20 comparable mapped fields; 18 omitted. Counts are not percentages.',
         fontsize=9, color='dimgray')
fig.tight_layout(rect=(0, 0.04, 1, 1))
output = project_root / 'notebooks' / 'exported_charts' / 'rq7_sensitive_disclosure_heatmap.png'
output.parent.mkdir(parents=True, exist_ok=True)
plt.savefig('exported_charts/rq7_sensitive_disclosure_heatmap.png', dpi=200, bbox_inches='tight')
static_output = project_root / 'backend' / 'static' / 'charts' / output.name
static_output.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(static_output, dpi=200, bbox_inches='tight')
print(f'Saved: {output}')
plt.show()


**Interpretation:** 14/16 personal-information, 3/3 finance-information, and 1/1 security-credential records are omitted; there are no comparable health-information records. These are small, manually mapped audit subsets, so the visual must not be read as an ecosystem-wide disclosure rate or a direct reproduction of the paper's full-category Figure 9.